In [1]:
import os, csv, time, random
from datetime import datetime
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ["Normal", "Benign", "Malignant"]
NUM_CLASSES = 3

BACKBONE_NAME = "ResNet50"
PRETRAINED = True
# INPUT_SIZE = (224, 224)   # change to (299, 299) later if needed
INPUT_SIZE = (299, 299)

# Experiment logging
EXPERIMENT_DIR = "experiments/resnet50_v1"
CSV_FILE = os.path.join(EXPERIMENT_DIR, "experiments.csv")
LOSS_PLOT_FOLDER = os.path.join(EXPERIMENT_DIR, "loss_plots")

os.makedirs(LOSS_PLOT_FOLDER, exist_ok=True)


2026-02-02 18:27:51.387121: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-02 18:27:51.405124: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-02 18:27:51.426569: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-02 18:27:51.433644: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-02 18:27:51.449104: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# Load balanced training data
X_train = np.load("images_bal.npy")
y_train = np.load("labels_bal.npy")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_train dtype:", X_train.dtype)
print("Label distribution:", np.bincount(y_train))

X_train shape: (9825, 299, 299, 1)
y_train shape: (9825,)
X_train dtype: float32
Label distribution: [3275 3275 3275]


In [3]:
# Load validation data
X_val = np.load("Kaggle_Data_recheck/Cleaned_Validation_Data/cv10_data.npy")
y_val = np.load("Kaggle_Data_recheck/Cleaned_Validation_Data/cv10_labels.npy")

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_val dtype:", X_val.dtype)
print("Val label distribution:", np.bincount(y_val))

X_val shape: (7682, 299, 299, 1)
y_val shape: (7682,)
X_val dtype: uint8
Val label distribution: [6663  593  426]


In [4]:
# Data preprocessing
def preprocess(x, y):
    x = tf.cast(x, tf.float32)
    x = tf.image.resize(x, INPUT_SIZE)
    x = tf.image.grayscale_to_rgb(x)
    x = preprocess_input(x)  # ImageNet normalization
    return x, y

In [5]:
# Data augmentation
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, 0.1)
    image = tf.image.random_contrast(image, 0.9, 1.1)
    return image, label

In [6]:
BATCH_SIZE = 32
SHUFFLE_BUFFER = 1000

# Training dataset
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(SHUFFLE_BUFFER, seed=SEED)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .map(augment, num_parallel_calls=tf.data.AUTOTUNE)  # ← added
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Validation dataset
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

I0000 00:00:1770056875.802037   11608 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1770056875.868409   11608 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1770056875.870895   11608 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1770056875.875903   11608 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

In [7]:
# Backbone
backbone = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(*INPUT_SIZE, 3)
)
backbone.trainable = False

# Classifier head
inputs = tf.keras.Input(shape=(*INPUT_SIZE, 3))
x = backbone(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 299, 299, 3)]     0         
                                                                 
 resnet50 (Functional)       (None, 10, 10, 2048)      23587712  
                                                                 
 global_average_pooling2d (  (None, 2048)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 512)               1049088   
                                                                 
 dropout (Dropout)           (None, 512)               0         
                                                                 
 dense_1 (Dense)             (None, 3)                 1539      
                                                             

In [8]:
# Compile
# LEARNING_RATE = 1e-3

# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"]
# )

# Unfreeze last ResNet50 block
# backbone.trainable = True
# for layer in backbone.layers[:-33]:  # freeze all except last conv block
#     layer.trainable = False

#     # Unfreeze conv5_x only
# for layer in backbone.layers:
#     layer.trainable = layer.name.startswith("conv5")

# Unfreeze conv4_x and conv5_x
for layer in backbone.layers:
    layer.trainable = (
        layer.name.startswith("conv3") or
        layer.name.startswith("conv4") or
        layer.name.startswith("conv5")
    )

for layer in backbone.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Focal Loss Function
def focal_loss_per_class(alpha=None, gamma=2.0):
    alpha = np.array(alpha if alpha is not None else [1.0]*NUM_CLASSES, dtype=np.float32)
    
    def loss_fn(y_true, y_pred):
        y_true_int = tf.cast(y_true, tf.int32)
        y_pred_soft = tf.nn.softmax(y_pred, axis=-1)
        y_pred_soft = tf.clip_by_value(y_pred_soft, 1e-7, 1-1e-7)
        
        # Probability of the true class
        pt = tf.gather(y_pred_soft, y_true_int[:, None], batch_dims=1)
        pt = tf.squeeze(pt, axis=1)
        
        # Class weights per sample
        alpha_t = tf.gather(alpha, y_true_int)
        
        # Focal loss
        loss = -alpha_t * (1 - pt)**gamma * tf.math.log(pt)
        return tf.reduce_mean(loss)
    
    return loss_fn

# Class weights for loss function
alpha_weights = [1.0, 1.0, 1.0]  # Normal, Benign, Malignant

# Compile with lower LR for fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    # loss="sparse_categorical_crossentropy",
    loss=focal_loss_per_class(alpha=alpha_weights, gamma=3.0),
    metrics=["accuracy"]
)

In [9]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

lr_reduce = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-6
)

callbacks = [early_stop, lr_reduce]

In [10]:
EPOCHS = 30

start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    # class_weight={0: 1.0, 1: 10.0, 2: 50.0},
    callbacks=callbacks,
    verbose=1
)

training_time_sec = time.time() - start_time
print(f"Training time (min): {training_time_sec / 60:.2f}")

Epoch 1/30


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
2026-02-02 18:28:11.205484: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90700
W0000 00:00:1770056891.289098   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056891.296994   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056891.299254   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056891.309141   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056891.311214   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056891.313937   11714 gpu_timer.cc:114] Skipping t

  1/308 [..............................] - ETA: 43:00 - loss: 0.3972 - accuracy: 0.2188

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
W0000 00:00:1770056894.834075   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056894.835681   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056894.837356   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056894.839160   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056894.840929   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056894.843036   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056894.846234   11714 gpu_timer.cc:114] Skipping

  2/308 [..............................] - ETA: 1:11 - loss: 0.3769 - accuracy: 0.2500 

...

308/308 [==============================] - ETA: 0s - loss: 0.3725 - accuracy: 0.3361 

W0000 00:00:1770056912.865521   11712 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056912.865677   11712 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056912.865784   11712 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056912.865902   11712 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056912.866080   11712 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056912.866185   11712 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056912.866287   11712 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056912.866485   11712 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056912.866619   11712 gp

308/308 [==============================] - 42s 108ms/step - loss: 0.3725 - accuracy: 0.3361 - val_loss: 0.2931 - val_accuracy: 0.5327 - lr: 1.0000e-06
Epoch 2/30
307/308 [============================>.] - ETA: 0s - loss: 0.3710 - accuracy: 0.3317 

W0000 00:00:1770056946.107295   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056946.107497   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056946.107647   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056946.107809   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056946.108027   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056946.108183   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056946.108338   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056946.108604   11714 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1770056946.108793   11714 gp

308/308 [==============================] - 32s 102ms/step - loss: 0.3710 - accuracy: 0.3317 - val_loss: 0.2819 - val_accuracy: 0.5896 - lr: 1.0000e-06
Epoch 3/30
308/308 [==============================] - 31s 101ms/step - loss: 0.3721 - accuracy: 0.3257 - val_loss: 0.2779 - val_accuracy: 0.6075 - lr: 1.0000e-06
Epoch 4/30
308/308 [==============================] - 31s 101ms/step - loss: 0.3673 - accuracy: 0.3356 - val_loss: 0.2764 - val_accuracy: 0.6117 - lr: 1.0000e-06
Epoch 5/30
308/308 [==============================] - 32s 103ms/step - loss: 0.3656 - accuracy: 0.3385 - val_loss: 0.2779 - val_accuracy: 0.6077 - lr: 1.0000e-06
Epoch 6/30
308/308 [==============================] - 32s 103ms/step - loss: 0.3677 - accuracy: 0.3334 - val_loss: 0.2727 - val_accuracy: 0.6338 - lr: 1.0000e-06
Epoch 7/30
308/308 [==============================] - 32s 103ms/step - loss: 0.3656 - accuracy: 0.3352 - val_loss: 0.2731 - val_accuracy: 0.6317 - lr: 1.0000e-06
Epoch 8/30
308/308 [===================

In [11]:
# Validation predictions
val_ds_eval = val_ds.unbatch().batch(32)
y_pred_probs = model.predict(val_ds_eval)
y_pred = np.argmax(y_pred_probs, axis=1)

# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
print("Confusion matrix:\n", cm)

# Normalized confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
print("\nNormalized confusion matrix:\n", np.round(cm_norm, 2))

# Classification report
report_dict = classification_report(
    y_val,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True
)
print("\nClassification report:\n",
      classification_report(
          y_val,
          y_pred,
          target_names=CLASS_NAMES
      )
)

# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)
for i, acc in enumerate(per_class_acc):
    print(f"Accuracy for class {i} ({CLASS_NAMES[i]}): {acc:.4f}")

241/241 [==============================] - 15s 58ms/step
Confusion matrix:
 [[4632  307 1724]
 [ 316   17  260]
 [ 196   10  220]]

Normalized confusion matrix:
 [[0.7  0.05 0.26]
 [0.53 0.03 0.44]
 [0.46 0.02 0.52]]

Classification report:
               precision    recall  f1-score   support

      Normal       0.90      0.70      0.78      6663
      Benign       0.05      0.03      0.04       593
   Malignant       0.10      0.52      0.17       426

    accuracy                           0.63      7682
   macro avg       0.35      0.41      0.33      7682
weighted avg       0.79      0.63      0.69      7682

Accuracy for class 0 (Normal): 0.6952
Accuracy for class 1 (Benign): 0.0287
Accuracy for class 2 (Malignant): 0.5164


2026-02-02 18:34:19.800339: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-02-02 18:34:19.800484: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]


In [12]:
# Auto-increment experiment ID
def get_next_experiment_id(csv_file):
    if not os.path.exists(csv_file):
        return 1
    with open(csv_file, "r") as f:
        return sum(1 for _ in f)  # header + rows

# Extract max L1/L2 across layers
def extract_l1_l2(model):
    l1_vals = []
    l2_vals = []
    for layer in model.layers:
        reg = getattr(layer, "kernel_regularizer", None)
        if reg:
            if hasattr(reg, "l1") and reg.l1 is not None: 
                l1_vals.append(float(tf.keras.backend.get_value(reg.l1)))
            if hasattr(reg, "l2") and reg.l2 is not None:
                l2_vals.append(float(tf.keras.backend.get_value(reg.l2)))       
    return (
        max(l1_vals) if l1_vals else 0.0,
        max(l2_vals) if l2_vals else 0.0)

# Extract max dropout rate
def extract_max_dropout(model):
    rates = []
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Dropout):
            rates.append(float(layer.rate))
    return max(rates) if rates else 0.0

# Build experiment row
exp_id = get_next_experiment_id(CSV_FILE)
l1_val, l2_val = extract_l1_l2(model)
dropout_val = extract_max_dropout(model)
learning_rate_val = float(model.optimizer.learning_rate.numpy())
train_loss = history.history["loss"][-1]
val_loss = history.history["val_loss"][-1]
train_acc = history.history["accuracy"][-1]
val_acc = history.history["val_accuracy"][-1]

row = {
    "experiment_id": exp_id,
    "timestamp": datetime.now().isoformat(),
    "seed": SEED,
    "backbone": BACKBONE_NAME,
    "pretrained": PRETRAINED,
    "input_size": f"{INPUT_SIZE[0]}x{INPUT_SIZE[1]}",
    "batch_size": BATCH_SIZE,
    "shuffle_buffer": SHUFFLE_BUFFER,
    "epochs": len(history.history["loss"]),
    "learning_rate": learning_rate_val,
    "dropout_rate": dropout_val,
    "l1": l1_val,
    "l2": l2_val,
    "train_loss": train_loss,
    "val_loss": val_loss,
    "train_accuracy": train_acc,
    "val_accuracy": val_acc,
    "training_time_min": training_time_sec / 60.0
}

for i, cls in enumerate(CLASS_NAMES):
    cls_l = cls.lower()
    row[f"acc_{cls_l}"] = per_class_acc[i]
    row[f"precision_{cls_l}"] = report_dict[cls]["precision"]
    row[f"recall_{cls_l}"] = report_dict[cls]["recall"]
    row[f"f1_{cls_l}"] = report_dict[cls]["f1-score"]

file_exists = os.path.isfile(CSV_FILE)
with open(CSV_FILE, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)

print(f"Experiment {exp_id} logged to {CSV_FILE}")

# Save loss plot
plt.figure(figsize=(8, 6))
plt.plot(history.history["loss"], label="Training Loss", marker='o')
plt.plot(history.history["val_loss"], label="Validation Loss", marker='o')
plt.title(f"Experiment {exp_id:05d} Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plot_filename = os.path.join(LOSS_PLOT_FOLDER, f"{exp_id:05d}.png")
plt.savefig(plot_filename)
plt.close()
print(f"Loss plot saved to {plot_filename}")


Experiment 32 logged to experiments/resnet50_v1/experiments.csv
Loss plot saved to experiments/resnet50_v1/loss_plots/00032.png
